# ResistAI Sentinel — original exploration notebook (archived)

This is the Google Colab notebook the project grew out of. It is kept
for provenance only. **Do not run it and do not build on it.**

The production code lives in `src/resistai/` and is what `app.py` runs.

Known problems in this notebook, all fixed in the refactor (see the
*What changed from the notebook* section of the README):

- **Target leakage** — `amr_risk_prob` was both a model feature and the
  source of the label, so reported accuracy was meaningless.
- **Non-deterministic IDs** — `uuid4()` per rerun and salted `hash()`
  for room assignment.
- **A hardcoded ngrok auth token** — redacted here; the original has been
  revoked. Outputs have been stripped for the same reason.
- **~12 competing versions of `app.py`** across 75 cells.


In [ ]:
import numpy as np, pandas as pd, random
from pathlib import Path

random.seed(42); np.random.seed(42)
N = 4000
Path("data").mkdir(exist_ok=True)

def doctor_note(flag):
    base = ["Patient under evaluation.", "Vitals monitored.", "Empiric antibiotics started."]
    amr = [
        "No improvement despite antibiotics.",
        "Suspected antimicrobial resistance.",
        "Empiric therapy failed.",
        "Recommend culture and antibiotic review."
    ]
    return " ".join(base + ([random.choice(amr)] if flag else []))

rows=[]
for i in range(N):
    broad = int(random.random()<0.4)
    reserved = int(random.random()<(0.2 if broad else 0.05))
    switch = int(np.random.poisson(1))
    icu = int(random.random()<0.2)
    fever = int(random.random()<0.25)
    wbc_high = int(random.random()<0.22)

    prob = 0.1 + 0.15*broad + 0.2*reserved + 0.1*switch + 0.15*icu + 0.1*(fever and wbc_high)
    prob = min(prob,0.85)

    level = "High" if prob>0.45 else "Moderate" if prob>0.25 else "Low"

    rows.append({
        "patient_id": f"P{i+1}",
        "broad_spectrum_used": broad,
        "reserved_abx_used": reserved,
        "antibiotic_switches": switch,
        "icu_admission": icu,
        "fever": fever,
        "wbc_high": wbc_high,
        "amr_risk_prob": round(prob,2),
        "amr_risk_level": level,
        "doctor_note": doctor_note(prob>0.35)
    })

df=pd.DataFrame(rows)
df.to_csv("data/amr_dataset.csv",index=False)
print("Dataset ready:",df.shape)


In [ ]:
def score(row):
    s=0
    s+=2 if row["broad_spectrum_used"] else 0
    s+=3 if row["reserved_abx_used"] else 0
    s+=1 if row["antibiotic_switches"]>=1 else 0
    s+=2 if row["icu_admission"] else 0
    s+=1 if row["fever"] and row["wbc_high"] else 0
    return s

def level(score):
    return "High" if score>=6 else "Moderate" if score>=3 else "Low"


In [ ]:
KEYWORDS=[
    "no improvement",
    "suspected antimicrobial resistance",
    "empiric therapy failed",
    "culture",
    "review antibiotic"
]

def flag(note:str)->int:
    note=note.lower()
    return int(any(k in note for k in KEYWORDS))


In [ ]:
!mkdir -p ml

In [ ]:
import pandas as pd, joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

df=pd.read_csv("data/amr_dataset.csv")
X=df.drop(columns=["patient_id","doctor_note","amr_risk_level"])
y=df["amr_risk_level"]

Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,random_state=42)
model=RandomForestClassifier(n_estimators=200,random_state=42)
model.fit(Xtr,ytr)

joblib.dump(model,"ml/amr_model.pkl")
print("Model trained & saved")


In [ ]:
!pip install streamlit
!pip install plotly
!pip install joblib
import streamlit as st, pandas as pd, joblib, plotly.express as px
# from scoring.risk_scoring import score, level  # Remove this line
# from nlp.notes_flags import flag              # Remove this line

st.set_page_config(page_title="ResistAI Sentinel", layout="wide")
st.title("🛡️ ResistAI Sentinel")
st.caption("Early AMR Risk Detection & Antibiotic Stewardship Intelligence")

df=pd.read_csv("data/amr_dataset.csv")
model=joblib.load("ml/amr_model.pkl")

left, right = st.columns([1,2])

with left:
    pid=st.selectbox("Patient",df["patient_id"])
    row=df[df["patient_id"]==pid].iloc[0]

    rule_score=score(row)
    rule_level=level(rule_score)
    ml_pred=model.predict([row.drop(["patient_id","doctor_note","amr_risk_level"])])[0]

    st.metric("Rule-based Risk",rule_level)
    st.metric("ML Prediction",ml_pred)
    st.metric("Risk Probability",row["amr_risk_prob"])

with right:
    st.subheader("Clinical Explanation")
    reasons=[]
    if row["reserved_abx_used"]: reasons.append("Reserved antibiotic used")
    if row["broad_spectrum_used"]: reasons.append("Broad-spectrum antibiotic")
    if row["icu_admission"]: reasons.append("ICU admission")
    if flag(row["doctor_note"]): reasons.append("Doctor note flags resistance")

    for r in reasons: st.warning(r)

    fig=px.histogram(df,x="amr_risk_level",color="amr_risk_level",
                     title="AMR Risk Distribution")
    st.plotly_chart(fig,use_container_width=True)

st.markdown("---")
st.success("✔️ System running in Explainable + ML hybrid mode")

In [ ]:
!ls


In [ ]:
!streamlit run app.py


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib

st.set_page_config(page_title="ResistAI Sentinel", layout="wide")

st.title("🛡️ ResistAI Sentinel")
st.caption("Early AMR Risk Detection & Antibiotic Stewardship Intelligence")

df = pd.read_csv("data/amr_dataset.csv")
model = joblib.load("ml/amr_model.pkl")

patient_id = st.selectbox("Select Patient", df["patient_id"])
row = df[df["patient_id"] == patient_id].iloc[0]

X = row.drop(["patient_id","doctor_note","amr_risk_level"])
pred = model.predict([X])[0]

st.metric("Predicted AMR Risk", pred)
st.metric("Risk Probability", row["amr_risk_prob"])

st.subheader("Doctor Notes")
st.write(row["doctor_note"])


In [ ]:
!streamlit run app.py

In [ ]:
!pip install -q pyngrok

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("REDACTED_SEE_ENV_NGROK_AUTHTOKEN")

In [ ]:
from pyngrok import ngrok
import time

time.sleep(5)
public_url = ngrok.connect(8501)
print("افتح هذا الرابط فقط:")
print(public_url)


In [ ]:
import os

os.system("""
streamlit run app.py \
  --server.port 8501 \
  --server.address 0.0.0.0 \
  --server.headless true \
  --browser.gatherUsageStats false &
""")


In [ ]:
import time
time.sleep(8)

In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print("✅ افتح هذا الرابط فقط:")
print(public_url)


In [ ]:
!pip install -q reportlab

In [ ]:
import streamlit as st
import pandas as pd
import joblib
import plotly.express as px
import numpy as np
def score(row):
    s = 0
    s += 2 if row["broad_spectrum_used"] else 0
    s += 3 if row["reserved_abx_used"] else 0
    s += 1 if row["antibiotic_switches"] >= 1 else 0
    s += 2 if row["icu_admission"] else 0
    s += 1 if row["fever"] and row["wbc_high"] else 0
    return s

def level(score):
    return "High" if score >= 6 else "Moderate" if score >= 3 else "Low"

KEYWORDS = [
    "no improvement",
    "suspected antimicrobial resistance",
    "empiric therapy failed",
    "culture",
    "review antibiotic"
]

def flag(note: str) -> int:
    note = note.lower()
    return int(any(k in note for k in KEYWORDS))

from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas


In [ ]:
st.set_page_config(
    page_title="ResistAI Sentinel",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.title("🛡️ ResistAI Sentinel")
st.caption("Early AMR Risk Detection & Antibiotic Stewardship Intelligence")

df = pd.read_csv("data/amr_dataset.csv")
model = joblib.load("ml/amr_model.pkl")


In [ ]:
patient_id = st.selectbox("Select Patient", df["patient_id"])
row = df[df["patient_id"] == patient_id].iloc[0]

NUMERIC_FEATURES = [
    "broad_spectrum_used",
    "reserved_abx_used",
    "antibiotic_switches",
    "icu_admission",
    "fever",
    "wbc_high",
    "amr_risk_prob"
]


X = pd.DataFrame([row[NUMERIC_FEATURES]])
pred = model.predict(X)[0]

rule_score = score(row)
rule_level = level(rule_score)

In [ ]:
col1, col2, col3 = st.columns(3)

col1.metric("Rule-based Risk", rule_level)
col2.metric("ML Prediction", pred)
col3.metric("Risk Probability", row["amr_risk_prob"])


In [ ]:
if "department" not in df.columns:
    st.warning("Department data not available")
else:
    heat_df = (
        df[df["amr_risk_level"] == "High"]
        .groupby("department")
        .size()
        .reset_index(name="high_risk_cases")
    )

    fig_heat = px.imshow(
        [heat_df["high_risk_cases"]],
        labels=dict(x="Department", color="High Risk Density"),
        x=heat_df["department"],
        y=["AMR Risk"],
        color_continuous_scale="Reds"
    )

    st.plotly_chart(fig_heat, use_container_width=True)


In [ ]:
import pandas as pd
import random

df = pd.read_csv("data/amr_dataset.csv")

departments = ["ICU", "ER", "Ward A", "Ward B", "Surgery", "Isolation"]

df["department"] = [random.choice(departments) for _ in range(len(df))]

df.to_csv("data/amr_dataset.csv", index=False)

print("✅ Department column added")
print(df.columns)


In [ ]:
st.markdown("## 🏥 Hospital Risk Heatmap")

heat_df = (
    df[df["amr_risk_level"] == "High"]
    .groupby("department")
    .size()
    .reset_index(name="high_risk_cases")
)

fig_heat = px.imshow(
    [heat_df["high_risk_cases"]],
    labels=dict(x="Department", color="High Risk Density"),
    x=heat_df["department"],
    y=["AMR Risk"],
    color_continuous_scale="Reds"
)

st.plotly_chart(fig_heat, use_container_width=True)


In [ ]:
st.markdown("## ⏱️ AMR Risk Timeline")

timeline = pd.DataFrame({
    "day": list(range(1, 8)),
    "risk_probability": np.clip(
        np.cumsum(np.random.normal(0.02, 0.05, 7)) + row["amr_risk_prob"],
        0, 1
    )
})

fig_time = px.line(
    timeline,
    x="day",
    y="risk_probability",
    markers=True,
    title="7-Day AMR Risk Trend"
)

st.plotly_chart(fig_time, use_container_width=True)


In [ ]:
st.markdown("## 🔍 Model Explainability")

importances = model.feature_importances_

fi_df = pd.DataFrame({
    "Feature": NUMERIC_FEATURES,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

fig_fi = px.bar(
    fi_df,
    x="Importance",
    y="Feature",
    orientation="h",
    title="Feature Importance (Explainable AI)",
    color="Importance",
    color_continuous_scale="Viridis"
)

st.plotly_chart(fig_fi, use_container_width=True)


In [ ]:
st.markdown("## 🧠 Clinical Explanation")

reasons = []
if row["reserved_abx_used"]: reasons.append("Reserved antibiotic used")
if row["broad_spectrum_used"]: reasons.append("Broad-spectrum antibiotic")
if row["icu_admission"]: reasons.append("ICU admission")
if flag(row["doctor_note"]): reasons.append("Doctor note flags resistance")

for r in reasons:
    st.warning(r)


In [ ]:
st.markdown("## 📄 Generate AMR Report")

if st.button("Generate PDF Report"):
    filename = f"AMR_Report_{patient_id}.pdf"
    c = canvas.Canvas(filename, pagesize=A4)

    c.setFont("Helvetica-Bold", 16)
    c.drawString(50, 800, "ResistAI Sentinel – AMR Risk Report")

    c.setFont("Helvetica", 12)
    c.drawString(50, 760, f"Patient ID: {patient_id}")
    c.drawString(50, 740, f"Predicted Risk: {pred}")
    c.drawString(50, 720, f"Risk Probability: {row['amr_risk_prob']}")

    c.drawString(50, 690, "Clinical Notes:")
    c.drawString(50, 670, row["doctor_note"][:100])

    c.drawString(50, 630, "Key Risk Drivers:")
    y = 610
    for r in reasons:
        c.drawString(60, y, f"- {r}")
        y -= 20

    c.save()
    st.success(f"PDF Generated: {filename}")


In [ ]:
import os

os.system("""
streamlit run app.py \
  --server.port 8501 \
  --server.address 0.0.0.0 \
  --server.headless true \
  --browser.gatherUsageStats false &
""")


In [ ]:
import time
time.sleep(8)


In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print("🔥 افتح هذا الرابط فقط:")
print(public_url)



In [ ]:
# =========================
# ResistAI Sentinel - FULL APP (ONE CELL)
# =========================

# -------- Imports --------
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import plotly.express as px
from reportlab.pdfgen import canvas
from pathlib import Path

# -------- Page Config --------
st.set_page_config(
    page_title="ResistAI Sentinel",
    layout="wide",
    initial_sidebar_state="expanded"
)

# -------- Load Data & Model --------
df = pd.read_csv("data/amr_dataset.csv")
model = joblib.load("ml/amr_model.pkl")

# -------- Header --------
st.title("🛡️ ResistAI Sentinel")
st.caption("Early AMR Risk Detection & Antibiotic Stewardship Intelligence")

# -------- Patient Selection --------
patient_id = st.selectbox("Select Patient", df["patient_id"])
row = df[df["patient_id"] == patient_id].iloc[0]

# -------- Prediction --------
NUMERIC_FEATURES = [
    "broad_spectrum_used",
    "reserved_abx_used",
    "antibiotic_switches",
    "icu_admission",
    "fever",
    "wbc_high",
    "amr_risk_prob"
]

X = pd.DataFrame([row[NUMERIC_FEATURES]])
pred = model.predict(X)[0]

# -------- Main Metrics --------
col1, col2 = st.columns(2)
col1.metric("Predicted AMR Risk", pred)
col2.metric("Risk Probability", row["amr_risk_prob"])

# -------- Doctor Notes --------
st.subheader("📝 Doctor Notes")
st.write(row["doctor_note"])

if "resistance" in row["doctor_note"].lower():
    st.warning("⚠️ Doctor note indicates possible antimicrobial resistance")

st.divider()

tab1, tab2, tab3, tab4 = st.tabs([
    "🔥 Heatmap",
    "📈 Timeline",
    "🧠 Explainability",
    "📄 PDF Report"
])

# =====================
# TAB 1: HEATMAP
# =====================
with tab1:
    st.subheader("AMR Hotspots by Department")

    if "department" in df.columns:
        heat_df = (
            df[df["amr_risk_level"] == "High"]
            .groupby("department")
            .size()
            .reset_index(name="High Risk Cases")
        )

        fig_heat = px.bar(
            heat_df,
            x="department",
            y="High Risk Cases",
            color="High Risk Cases",
            title="Hospital AMR Heatmap"
        )
        st.plotly_chart(fig_heat, use_container_width=True)
    else:
        st.warning("No department data available in dataset")

# =====================
# TAB 2: TIMELINE
# =====================
with tab2:
    st.subheader("Patient AMR Risk Over Time")

    df_patient = df[df["patient_id"] == patient_id].copy()
    df_patient["step"] = range(len(df_patient))

    fig_time = px.line(
        df_patient,
        x="step",
        y="amr_risk_prob",
        markers=True
    )
    st.plotly_chart(fig_time, use_container_width=True)

# =====================
# TAB 3: EXPLAINABILITY
# =====================
with tab3:
    st.subheader("Model Feature Importance")

    fi_df = pd.DataFrame({
        "Feature": NUMERIC_FEATURES,
        "Importance": model.feature_importances_
    }).sort_values(by="Importance", ascending=False)

    fig_fi = px.bar(
        fi_df,
        x="Importance",
        y="Feature",
        orientation="h"
    )
    st.plotly_chart(fig_fi, use_container_width=True)

# =====================
# TAB 4: PDF REPORT
# =====================
with tab4:
    st.subheader("Generate Patient PDF Report")

    if st.button("Generate PDF"):
        Path("reports").mkdir(exist_ok=True)
        file_path = f"reports/{patient_id}_AMR_Report.pdf"

        c = canvas.Canvas(file_path)
        c.drawString(50, 800, "ResistAI Sentinel - AMR Report")
        c.drawString(50, 770, f"Patient ID: {patient_id}")
        c.drawString(50, 750, f"Risk Level: {pred}")
        c.drawString(50, 730, f"Risk Probability: {row['amr_risk_prob']}")
        c.drawString(50, 700, "Doctor Notes:")
        c.drawString(50, 680, row["doctor_note"][:100])
        c.save()

        st.success(f"PDF generated: {file_path}")


# =========================
# 1️⃣ HEATMAP (Department)
# =========================
st.subheader("🔥 AMR Heatmap by Department")

if "department" in df.columns:
    heat_df = (
        df[df["amr_risk_level"] == "High"]
        .groupby("department")
        .size()
        .reset_index(name="High Risk Cases")
    )

    fig_heat = px.bar(
        heat_df,
        x="department",
        y="High Risk Cases",
        color="High Risk Cases",
        title="Hospital AMR Hotspots"
    )
    st.plotly_chart(fig_heat, use_container_width=True)
else:
    st.info("ℹ️ Column 'department' not found in dataset")

# =========================
# 2️⃣ TIMELINE
# =========================
st.subheader("📈 AMR Risk Timeline")

df["time_index"] = range(len(df))
fig_time = px.line(
    df[df["patient_id"] == patient_id],
    x="time_index",
    y="amr_risk_prob",
    markers=True,
    title="AMR Risk Progression Over Time"
)
st.plotly_chart(fig_time, use_container_width=True)

# =========================
# 3️⃣ FEATURE IMPORTANCE
# =========================
st.subheader("🧠 Model Explainability (Feature Importance)")

fi_df = pd.DataFrame({
    "Feature": NUMERIC_FEATURES,
    "Importance": model.feature_importances_
}).sort_values(by="Importance", ascending=False)

fig_fi = px.bar(
    fi_df,
    x="Importance",
    y="Feature",
    orientation="h",
    title="Feature Importance"
)
st.plotly_chart(fig_fi, use_container_width=True)

# =========================
# 4️⃣ RULE-BASED SIGNALS
# =========================
st.subheader("⚖️ Rule-based Clinical Signals")

if row["reserved_abx_used"]:
    st.error("Reserved antibiotic used")
if row["broad_spectrum_used"]:
    st.warning("Broad-spectrum antibiotic used")
if row["icu_admission"]:
    st.warning("ICU admission")
if row["fever"] and row["wbc_high"]:
    st.warning("Fever with elevated WBC")

# =========================
# 5️⃣ HOSPITAL KPIs
# =========================
st.subheader("📊 Hospital Overview")

k1, k2, k3 = st.columns(3)
k1.metric("High Risk Patients", len(df[df["amr_risk_level"] == "High"]))
k2.metric("ICU Admissions", df["icu_admission"].sum())
k3.metric("Broad-spectrum Usage", df["broad_spectrum_used"].sum())

# =========================
# 6️⃣ ALERT SYSTEM
# =========================
if pred == "High":
    st.error("🚨 High AMR Risk — Immediate Infection Control Recommended")

# =========================
# 7️⃣ PDF REPORT
# =========================
st.subheader("📄 Generate Patient Report")

if st.button("Generate PDF Report"):
    Path("reports").mkdir(exist_ok=True)
    file_path = f"reports/{patient_id}_AMR_Report.pdf"

    c = canvas.Canvas(file_path)
    c.drawString(50, 800, "ResistAI Sentinel - AMR Risk Report")
    c.drawString(50, 770, f"Patient ID: {patient_id}")
    c.drawString(50, 750, f"Predicted Risk: {pred}")
    c.drawString(50, 730, f"Risk Probability: {row['amr_risk_prob']}")
    c.drawString(50, 700, "Doctor Notes:")
    c.drawString(50, 680, row["doctor_note"][:90])
    c.save()

    st.success(f"PDF Generated: {file_path}")

# =========================
# 8️⃣ DATA VIEW
# =========================
with st.expander("🔍 View Patient Raw Data"):
    st.dataframe(row.to_frame())

# =========================
# 9️⃣ FOOTER
# =========================
st.caption("© 2026 ResistAI Sentinel — Intelligent AMR Surveillance System")


In [ ]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0
!apt-get -qq install cloudflared
!cloudflared tunnel --url http://localhost:8501


In [ ]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 &


In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(8501, bind_tls=True)
print("🚀 افتح هذا الرابط:")
print(public_url)


In [ ]:
!pkill streamlit
!pkill ngrok


In [ ]:
!ls


In [ ]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > streamlit.log 2>&1 &

In [ ]:
!curl http://localhost:8501


In [ ]:
from pyngrok import ngrok

ngrok.kill()  # تنظيف
public_url = ngrok.connect(8501, "http")
print("🔥 افتح الرابط:")
print(public_url)


In [ ]:
st.divider()

tab1, tab2, tab3, tab4 = st.tabs([
    "🔥 Heatmap",
    "📈 Timeline",
    "🧠 Explainability",
    "📄 PDF Report"
])

# =====================
# TAB 1: HEATMAP
# =====================
with tab1:
    st.subheader("AMR Hotspots by Department")

    if "department" in df.columns:
        heat_df = (
            df[df["amr_risk_level"] == "High"]
            .groupby("department")
            .size()
            .reset_index(name="High Risk Cases")
        )

        fig_heat = px.bar(
            heat_df,
            x="department",
            y="High Risk Cases",
            color="High Risk Cases",
            title="Hospital AMR Heatmap"
        )
        st.plotly_chart(fig_heat, use_container_width=True)
    else:
        st.warning("No department data available in dataset")

# =====================
# TAB 2: TIMELINE
# =====================
with tab2:
    st.subheader("Patient AMR Risk Over Time")

    df_patient = df[df["patient_id"] == patient_id].copy()
    df_patient["step"] = range(len(df_patient))

    fig_time = px.line(
        df_patient,
        x="step",
        y="amr_risk_prob",
        markers=True
    )
    st.plotly_chart(fig_time, use_container_width=True)

# =====================
# TAB 3: EXPLAINABILITY
# =====================
with tab3:
    st.subheader("Model Feature Importance")

    fi_df = pd.DataFrame({
        "Feature": NUMERIC_FEATURES,
        "Importance": model.feature_importances_
    }).sort_values(by="Importance", ascending=False)

    fig_fi = px.bar(
        fi_df,
        x="Importance",
        y="Feature",
        orientation="h"
    )
    st.plotly_chart(fig_fi, use_container_width=True)

# =====================
# TAB 4: PDF REPORT
# =====================
with tab4:
    st.subheader("Generate Patient PDF Report")

    if st.button("Generate PDF"):
        Path("reports").mkdir(exist_ok=True)
        file_path = f"reports/{patient_id}_AMR_Report.pdf"

        c = canvas.Canvas(file_path)
        c.drawString(50, 800, "ResistAI Sentinel - AMR Report")
        c.drawString(50, 770, f"Patient ID: {patient_id}")
        c.drawString(50, 750, f"Risk Level: {pred}")
        c.drawString(50, 730, f"Risk Probability: {row['amr_risk_prob']}")
        c.drawString(50, 700, "Doctor Notes:")
        c.drawString(50, 680, row["doctor_note"][:100])
        c.save()

        st.success(f"PDF generated: {file_path}")


In [ ]:
!pkill streamlit
!pkill ngrok


In [ ]:
!pwd
!ls


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import plotly.express as px
from reportlab.pdfgen import canvas
from pathlib import Path

st.set_page_config(page_title="ResistAI Sentinel", layout="wide")

df = pd.read_csv("data/amr_dataset.csv")
model = joblib.load("ml/amr_model.pkl")

st.title("🛡️ ResistAI Sentinel")
st.caption("Early AMR Risk Detection & Antibiotic Stewardship Intelligence")

patient_id = st.selectbox("Select Patient", df["patient_id"])
row = df[df["patient_id"] == patient_id].iloc[0]

NUMERIC_FEATURES = [
    "broad_spectrum_used",
    "reserved_abx_used",
    "antibiotic_switches",
    "icu_admission",
    "fever",
    "wbc_high",
    "amr_risk_prob"
]

X = pd.DataFrame([row[NUMERIC_FEATURES]])
pred = model.predict(X)[0]

st.metric("Predicted AMR Risk", pred)
st.metric("Risk Probability", row["amr_risk_prob"])

st.subheader("Doctor Notes")
st.write(row["doctor_note"])

# =======================
# TABS (إجباري للظهور)
# =======================
tab1, tab2, tab3, tab4 = st.tabs([
    "🔥 Heatmap",
    "📈 Timeline",
    "🧠 Explainability",
    "📄 PDF"
])

with tab1:
    st.write("🔥 HEATMAP TEST")
    if "department" in df.columns:
        heat = df.groupby("department").size().reset_index(name="count")
        st.plotly_chart(px.bar(heat, x="department", y="count"))
    else:
        st.error("❌ No department column")

with tab2:
    st.write("📈 TIMELINE TEST")
    st.line_chart(df[df["patient_id"] == patient_id]["amr_risk_prob"])

with tab3:
    st.write("🧠 FEATURE IMPORTANCE TEST")
    fi = pd.DataFrame({
        "feature": NUMERIC_FEATURES,
        "importance": model.feature_importances_
    })
    st.plotly_chart(px.bar(fi, x="importance", y="feature", orientation="h"))

with tab4:
    if st.button("Generate PDF"):
        Path("reports").mkdir(exist_ok=True)
        c = canvas.Canvas(f"reports/{patient_id}.pdf")
        c.drawString(50, 800, "ResistAI Sentinel Report")
        c.drawString(50, 770, f"Patient: {patient_id}")
        c.drawString(50, 750, f"Risk: {pred}")
        c.save()
        st.success("PDF Generated")


In [ ]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 &

In [ ]:
from pyngrok import ngrok
ngrok.kill()
url = ngrok.connect(8501, "http")
print(url)


In [ ]:
!pkill -9 -f streamlit
!pkill -9 -f ngrok
!rm -f streamlit.log

In [ ]:
!ls

In [ ]:
!nohup streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > streamlit.log 2>&1 &

In [ ]:
!sleep 3
!lsof -i :8501

In [ ]:
from pyngrok import ngrok

ngrok.kill()
public_url = ngrok.connect(8501, "http")
print(public_url)

In [ ]:
st.write("Available columns:", df.columns.tolist())

In [ ]:
# محاولة ذكية لاكتشاف عمود القسم
POSSIBLE_DEPT_COLUMNS = [
    "department",
    "ward",
    "unit",
    "service",
    "location",
    "dept"
]

dept_col = None
for col in POSSIBLE_DEPT_COLUMNS:
    if col in df.columns:
        dept_col = col
        break

if dept_col is None:
    st.error("❌ No department/ward column found in dataset")
    st.stop()

st.success(f"✅ Using '{dept_col}' as department column")


In [ ]:
st.write(df.columns.tolist())


In [ ]:
def assign_department(row):
    if row["icu_admission"] == 1:
        return "ICU"
    elif row["amr_risk_prob"] >= 0.7:
        return "Infectious Diseases"
    elif row["amr_risk_prob"] >= 0.4:
        return "General Ward"
    else:
        return "Outpatient"

df["department"] = df.apply(assign_department, axis=1)
dept_col = "department"


In [ ]:
st.markdown("## 🏥 Hospital AMR Risk by Department")

dept_df = (
    df.groupby(dept_col)
    .agg(
        avg_risk=("amr_risk_prob", "mean"),
        high_cases=("amr_risk_level", lambda x: (x == "High").sum()),
        total_cases=("patient_id", "count")
    )
    .reset_index()
)

import plotly.express as px

fig = px.bar(
    dept_df,
    x="avg_risk",
    y=dept_col,
    orientation="h",
    color="avg_risk",
    color_continuous_scale=["green", "yellow", "red"],
    title="AMR Risk Distribution Across Hospital Departments"
)

st.plotly_chart(fig, use_container_width=True)


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import plotly.express as px
from pathlib import Path

# ===============================
# Page Config
# ===============================
st.set_page_config(page_title="ResistAI Sentinel", layout="wide")

# ===============================
# Load Data & Model
# ===============================
df = pd.read_csv("data/amr_dataset.csv")
model = joblib.load("ml/amr_model.pkl")

# ===============================
# Header
# ===============================
st.title("🛡️ ResistAI Sentinel")
st.caption("Hospital-level AMR Risk Surveillance Dashboard")

# ===============================
# Patient Selector
# ===============================
patient_id = st.selectbox("Select Patient", df["patient_id"])
row = df[df["patient_id"] == patient_id].iloc[0]

NUMERIC_FEATURES = [
    "broad_spectrum_used",
    "reserved_abx_used",
    "antibiotic_switches",
    "icu_admission",
    "fever",
    "wbc_high",
    "amr_risk_prob"
]

X = pd.DataFrame([row[NUMERIC_FEATURES]])
pred = model.predict(X)[0]

c1, c2 = st.columns(2)
c1.metric("Predicted AMR Risk", pred)
c2.metric("Risk Probability", row["amr_risk_prob"])

st.subheader("Doctor Notes")
st.write(row["doctor_note"])

# ===============================
# Create Hospital Departments
# ===============================
def assign_department(r):
    if r["icu_admission"] == 1:
        return "ICU"
    elif r["amr_risk_prob"] >= 0.7:
        return "Infectious Diseases"
    elif r["amr_risk_prob"] >= 0.4:
        return "General Ward"
    else:
        return "ER"

df["department"] = df.apply(assign_department, axis=1)

# ===============================
# Department Risk + Colors
# ===============================
dept_risk = df.groupby("department")["amr_risk_prob"].mean().to_dict()

def risk_color(v):
    if v >= 0.7:
        return "#ff4d4d"
    elif v >= 0.4:
        return "#ffa500"
    return "#4caf50"

# ===============================
# TABS
# ===============================
tab1, tab2, tab3 = st.tabs([
    "🗺️ Hospital Map",
    "📊 Management Dashboard",
    "📈 Department Heatmap"
])

# ===============================
# TAB 1: Interactive Hospital Map
# ===============================
with tab1:
    st.subheader("Interactive Hospital Flow (ER → ICU → Ward)")
    cols = st.columns(4)
    for col, dept in zip(
        cols,
        ["ER", "ICU", "Infectious Diseases", "General Ward"]
    ):
        col.markdown(
            f"""
            <div style="
            background:{risk_color(dept_risk[dept])};
            padding:25px;
            border-radius:12px;
            text-align:center;
            color:white;
            font-weight:bold;">
            <h4>{dept}</h4>
            Avg Risk: {dept_risk[dept]:.2f}
            </div>
            """,
            unsafe_allow_html=True
        )

# ===============================
# TAB 2: Management Dashboard
# ===============================
with tab2:
    st.subheader("Top High-Risk Departments")

    summary = (
        df.groupby("department")
        .agg(
            avg_risk=("amr_risk_prob", "mean"),
            high_cases=("amr_risk_level", lambda x: (x == "High").sum()),
            patients=("patient_id", "count")
        )
        .reset_index()
        .sort_values("avg_risk", ascending=False)
    )

    st.dataframe(summary.head(5), use_container_width=True)

# ===============================
# TAB 3: Heatmap
# ===============================
with tab3:
    st.subheader("Hospital AMR Risk Heatmap")

    fig = px.bar(
        summary,
        x="avg_risk",
        y="department",
        orientation="h",
        color="avg_risk",
        color_continuous_scale=["green", "yellow", "red"],
        labels={"avg_risk": "Average Risk", "department": "Department"}
    )
    st.plotly_chart(fig, use_container_width=True)

st.caption("© 2026 ResistAI Sentinel")


In [ ]:
# ===============================
# CREATE DEPARTMENT
# ===============================
def assign_department(r):
    if r["icu_admission"] == 1:
        return "ICU"
    elif r["amr_risk_prob"] >= 0.7:
        return "Infectious Diseases"
    elif r["amr_risk_prob"] >= 0.4:
        return "General Ward"
    else:
        return "Outpatient"

df["department"] = df.apply(assign_department, axis=1)

# ===============================
# HOSPITAL MAP
# ===============================
HOSPITAL_MAP = {
    "ER": ["ER-1", "ER-2", "ER-Corridor"],
    "ICU": ["ICU-1", "ICU-2", "ICU-Corridor"],
    "Infectious Diseases": ["ID-1", "ID-2", "ID-Corridor"],
    "General Ward": ["GW-1", "GW-2", "GW-Corridor"],
    "Outpatient": ["OP-1", "OP-2", "OP-Corridor"]
}

# ===============================
# CREATE ROOM (THIS IS REQUIRED)
# ===============================
def assign_room(row):
    dept = row["department"]
    if dept not in HOSPITAL_MAP:
        dept = "ER"
    rooms = HOSPITAL_MAP[dept]
    return rooms[hash(row["patient_id"]) % len(rooms)]

df["room"] = df.apply(assign_room, axis=1)


In [ ]:
# ===============================
# DEPARTMENT SUMMARY (GLOBAL)
# ===============================
summary = (
    df.groupby("department")
    .agg(
        avg_risk=("amr_risk_prob", "mean"),
        high_cases=("amr_risk_level", lambda x: (x == "High").sum()),
        patients=("patient_id", "count")
    )
    .reset_index()
)


In [ ]:
# =====================================================
# HOSPITAL LAYOUT (Rooms + Corridors)
# =====================================================

HOSPITAL_MAP = {
    "ER": ["ER-1", "ER-2", "ER-Corridor"],
    "ICU": ["ICU-1", "ICU-2", "ICU-Corridor"],
    "Infectious Diseases": ["ID-1", "ID-2", "ID-Corridor"],
    "General Ward": ["GW-1", "GW-2", "GW-Corridor"],
    "Outpatient": ["OP-1", "OP-2", "OP-Corridor"]
}


# Assign room based on department + patient id (deterministic)
def assign_room(row):
    dept = row["department"]
    if dept not in HOSPITAL_MAP:
        dept = "ER"  # fallback آمن
    rooms = HOSPITAL_MAP[dept]
    return rooms[hash(row["patient_id"]) % len(rooms)]

# =====================================================
# TABS (EXTENDED)
# =====================================================
tab_map, tab_timeline, tab_heatmap, tab_actions = st.tabs([
    "🗺️ Hospital Layout",
    "⏱️ Patient Movement Timeline",
    "🔥 Hospital Heatmap",
    "🧠 Recommendations"
])

# =====================================================
# TAB 1: HOSPITAL LAYOUT (VISUAL MAP)
# =====================================================
with tab_map:
    st.subheader("Hospital Layout – Rooms & Corridors Risk Map")

    room_risk = (
        df.groupby("room")["amr_risk_prob"]
        .mean()
        .reset_index()
        .sort_values("amr_risk_prob", ascending=False)
    )

    fig_rooms = px.bar(
        room_risk,
        x="amr_risk_prob",
        y="room",
        orientation="h",
        color="amr_risk_prob",
        color_continuous_scale=["green", "yellow", "red"],
        labels={"amr_risk_prob": "Average Risk"}
    )

    st.plotly_chart(fig_rooms, use_container_width=True)

# =====================================================
# TAB 2: TIMELINE – PATIENT FLOW
# =====================================================
with tab_timeline:
    st.subheader("Patient Risk Timeline & Department Flow")

    patient_df = df[df["patient_id"] == patient_id].copy()
    patient_df["time_step"] = range(len(patient_df))

    fig_timeline = px.scatter(
        patient_df,
        x="time_step",
        y="department",
        size="amr_risk_prob",
        color="amr_risk_prob",
        color_continuous_scale=["green", "yellow", "red"],
        labels={"time_step": "Time", "department": "Department"},
        title="Patient Movement & Risk Over Time"
    )

    st.plotly_chart(fig_timeline, use_container_width=True)

# =====================================================
# TAB 3: FULL HOSPITAL HEATMAP (ROOMS + CORRIDORS)
# =====================================================
with tab_heatmap:
    st.subheader("Full Hospital Heatmap (Rooms & Corridors)")

    fig_heat = px.treemap(
        df,
        path=["department", "room"],
        values="amr_risk_prob",
        color="amr_risk_prob",
        color_continuous_scale=["green", "yellow", "red"],
        title="Hospital Infection Risk Heatmap"
    )

    st.plotly_chart(fig_heat, use_container_width=True)

# =====================================================
# TAB 4: AUTOMATED RECOMMENDATIONS & ACTIONS
# =====================================================
with tab_actions:
    st.subheader("Automated Infection Control Recommendations")

    high_rooms = room_risk[room_risk["amr_risk_prob"] >= 0.7]["room"].tolist()
    high_depts = summary[summary["avg_risk"] >= 0.7]["department"].tolist()

    if high_rooms:
        st.error("🚨 Immediate Room Disinfection Required")
        for r in high_rooms:
            st.write(f"- Disinfect **{r}** within 1 hour")

    if high_depts:
        st.warning("⚠️ Department-Level Actions")
        for d in high_depts:
            st.write(f"- Restrict admissions to **{d}**")
            st.write(f"- Review antibiotic usage in **{d}**")

    if not high_rooms and not high_depts:
        st.success("✅ No critical infection hotspots detected")

    st.markdown("### 🧾 Recommended Procedures")
    st.markdown("""
    - Increase environmental sampling
    - Enforce hand hygiene audits
    - Review antibiotic stewardship protocols
    - Isolate high-risk patients
    - Notify infection control team
    """)


In [ ]:
!pkill -9 -f streamlit
!pkill -9 -f ngrok
!rm -f streamlit.log


In [ ]:
%%writefile app.py
import streamlit as st

st.set_page_config(layout="wide")

st.title("🚨 STREAMLIT RELOAD TEST")
st.success("إذا تشوف هذا العنوان → السيرفر فعلاً أعاد التحميل")

st.write("Time:", __import__("time").time())


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import plotly.express as px

# ===============================
# BASIC CONFIG
# ===============================
st.set_page_config(page_title="ResistAI Sentinel", layout="wide")

# ===============================
# LOAD DATA
# ===============================
df = pd.read_csv("data/amr_dataset.csv")
model = joblib.load("ml/amr_model.pkl")

# ===============================
# HEADER
# ===============================
st.title("🛡️ ResistAI Sentinel")
st.caption("Hospital-level AMR Surveillance Dashboard")

# ===============================
# PATIENT SELECTION
# ===============================
patient_id = st.selectbox("Select Patient", df["patient_id"])
row = df[df["patient_id"] == patient_id].iloc[0]

# ===============================
# PREDICTION
# ===============================
NUMERIC_FEATURES = [
    "broad_spectrum_used",
    "reserved_abx_used",
    "antibiotic_switches",
    "icu_admission",
    "fever",
    "wbc_high",
    "amr_risk_prob"
]

X = pd.DataFrame([row[NUMERIC_FEATURES]])
pred = model.predict(X)[0]

c1, c2 = st.columns(2)
c1.metric("Predicted AMR Risk", pred)
c2.metric("Risk Probability", row["amr_risk_prob"])

st.subheader("Doctor Notes")
st.write(row["doctor_note"])

# ===============================
# ASSIGN DEPARTMENT
# ===============================
def assign_department(r):
    if r["icu_admission"] == 1:
        return "ICU"
    elif r["amr_risk_prob"] >= 0.7:
        return "Infectious Diseases"
    elif r["amr_risk_prob"] >= 0.4:
        return "General Ward"
    else:
        return "Outpatient"

df["department"] = df.apply(assign_department, axis=1)

# ===============================
# HOSPITAL MAP
# ===============================
HOSPITAL_MAP = {
    "ER": ["ER-1", "ER-2", "ER-Corridor"],
    "ICU": ["ICU-1", "ICU-2", "ICU-Corridor"],
    "Infectious Diseases": ["ID-1", "ID-2", "ID-Corridor"],
    "General Ward": ["GW-1", "GW-2", "GW-Corridor"],
    "Outpatient": ["OP-1", "OP-2", "OP-Corridor"]
}

def assign_room(r):
    dept = r["department"]
    if dept not in HOSPITAL_MAP:
        dept = "ER"
    rooms = HOSPITAL_MAP[dept]
    return rooms[hash(r["patient_id"]) % len(rooms)]

df["room"] = df.apply(assign_room, axis=1)
!pkill -9 -f streamlit
!pkill -9 -f ngrok
!rm -f streamlit.log
# ===============================
# GLOBAL SUMMARY
# ===============================
summary = (
    df.groupby("department")
    .agg(
        avg_risk=("amr_risk_prob", "mean"),
        high_cases=("amr_risk_level", lambda x: (x == "High").sum()),
        patients=("patient_id", "count")
    )
    .reset_index()
)

room_risk = (
    df.groupby("room")["amr_risk_prob"]
    .mean()
    .reset_index()
)

# ===============================
# TABS
# ===============================
tab1, tab2, tab3, tab4 = st.tabs([
    "🗺️ Hospital Map",
    "⏱️ Patient Timeline",
    "🔥 Full Heatmap",
    "🧠 Recommendations"
])

# ===============================
# TAB 1: HOSPITAL MAP
# ===============================
with tab1:
    st.subheader("Hospital Departments Risk Status")
    fig1 = px.bar(
        summary,
        x="avg_risk",
        y="department",
        orientation="h",
        color="avg_risk",
        color_continuous_scale=["green", "yellow", "red"]
    )
    st.plotly_chart(fig1, use_container_width=True)

# ===============================
# TAB 2: TIMELINE
# ===============================
with tab2:
    st.subheader("Patient Risk Timeline")
    patient_df = df[df["patient_id"] == patient_id].copy()
    patient_df["time_step"] = range(len(patient_df))

    fig2 = px.scatter(
        patient_df,
        x="time_step",
        y="department",
        size="amr_risk_prob",
        color="amr_risk_prob",
        color_continuous_scale=["green", "yellow", "red"]
    )
    st.plotly_chart(fig2, use_container_width=True)

# ===============================
# TAB 3: FULL HEATMAP
# ===============================
with tab3:
    st.subheader("Rooms & Corridors Heatmap")
    fig3 = px.treemap(
        df,
        path=["department", "room"],
        values="amr_risk_prob",
        color="amr_risk_prob",
        color_continuous_scale=["green", "yellow", "red"]
    )
    st.plotly_chart(fig3, use_container_width=True)

# ===============================
# TAB 4: RECOMMENDATIONS
# ===============================
with tab4:
    st.subheader("Automated Infection Control Actions")

    high_rooms = room_risk[room_risk["amr_risk_prob"] >= 0.7]["room"].tolist()
    high_depts = summary[summary["avg_risk"] >= 0.7]["department"].tolist()

    if high_rooms:
        st.error("🚨 Immediate Room Disinfection Required")
        for r in high_rooms:
            st.write(f"- Disinfect {r}")

    if high_depts:
        st.warning("⚠️ Department-Level Actions")
        for d in high_depts:
            st.write(f"- Restrict admissions to {d}")
            st.write(f"- Review antibiotics usage in {d}")

    if not high_rooms and not high_depts:
        st.success("✅ No critical hotspots detected")

st.caption("© 2026 ResistAI Sentinel")


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import plotly.express as px
import uuid

# ===============================
# PAGE CONFIG
# ===============================
st.set_page_config(page_title="ResistAI Sentinel", layout="wide")

# ===============================
# LOAD DATA & MODEL
# ===============================
df = pd.read_csv("data/amr_dataset.csv")
model = joblib.load("ml/amr_model.pkl")

# ===============================
# CREATE MEDICAL PATIENT ID
# ===============================
df["patient_uid"] = df["patient_id"].apply(
    lambda _: f"MRN-{uuid.uuid4().hex[:8].upper()}"
)

# ===============================
# HEADER
# ===============================
st.title("🛡️ ResistAI Sentinel")
st.caption("Explainable AMR Risk Intelligence for Hospitals")

# ===============================
# EXECUTIVE DASHBOARD (POWER BI STYLE)
# ===============================
st.markdown("## 📊 Executive Overview")

c1, c2, c3, c4 = st.columns(4)
c1.metric("Total Patients", len(df))
c2.metric("High Risk Patients", (df["amr_risk_level"] == "High").sum())
c3.metric("ICU Admissions", df["icu_admission"].sum())
c4.metric("Avg Hospital Risk", round(df["amr_risk_prob"].mean(), 2))

# ===============================
# PATIENT SELECTION
# ===============================
st.markdown("## 🧑‍⚕️ Patient Analysis")

patient_uid = st.selectbox("Select Patient (Medical ID)", df["patient_uid"])
row = df[df["patient_uid"] == patient_uid].iloc[0]

# ===============================
# PREDICTION
# ===============================
FEATURES = [
    "broad_spectrum_used",
    "reserved_abx_used",
    "antibiotic_switches",
    "icu_admission",
    "fever",
    "wbc_high",
    "amr_risk_prob"
]

X = pd.DataFrame([row[FEATURES]])
prediction = model.predict(


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import plotly.express as px
import uuid

# ===============================
# PAGE CONFIG
# ===============================
st.set_page_config(page_title="ResistAI Sentinel", layout="wide")

# ===============================
# LOAD DATA & MODEL
# ===============================
df = pd.read_csv("data/amr_dataset.csv")
model = joblib.load("ml/amr_model.pkl")

# ===============================
# CREATE MEDICAL PATIENT ID
# ===============================
df["patient_uid"] = df["patient_id"].apply(
    lambda _: f"MRN-{uuid.uuid4().hex[:8].upper()}"
)

# ===============================
# HEADER
# ===============================
st.title("🛡️ ResistAI Sentinel")
st.caption("Explainable AMR Risk Intelligence for Hospitals")

# ===============================
# EXECUTIVE DASHBOARD
# ===============================
st.markdown("## 📊 Executive Overview")

c1, c2, c3, c4 = st.columns(4)
c1.metric("Total Patients", len(df))
c2.metric("High Risk Patients", (df["amr_risk_level"] == "High").sum())
c3.metric("ICU Admissions", df["icu_admission"].sum())
c4.metric("Avg Hospital Risk", round(df["amr_risk_prob"].mean(), 2))

# ===============================
# PATIENT SELECTION
# ===============================
st.markdown("## 🧑‍⚕️ Patient Analysis")

patient_uid = st.selectbox("Select Patient (Medical ID)", df["patient_uid"])
row = df[df["patient_uid"] == patient_uid].iloc[0]
# ===============================
# 1️⃣ RISK TREND & EARLY WARNING
# ===============================
import numpy as np

# Simulate last 5 risk readings
np.random.seed(0)
history = np.clip(
    row["amr_risk_prob"] + np.random.normal(0, 0.05, 5),
    0, 1
)

trend_score = history[-3:].mean() - history[:3].mean()

if trend_score > 0.05:
    trend_label = "Rising"
    trend_color = "🔴"
elif trend_score < -0.05:
    trend_label = "Decreasing"
    trend_color = "🟢"
else:
    trend_label = "Stable"
    trend_color = "🟡"

st.markdown("### 🧠 Risk Trend")
st.metric("Trend Status", f"{trend_color} {trend_label}")
st.line_chart(history)

# ===============================
# 2️⃣ WHAT-IF SIMULATOR
# ===============================
st.markdown("### 🧭 What-If Simulator")

c1, c2, c3 = st.columns(3)
sim_broad = c1.checkbox("Use Broad-Spectrum ABX", value=bool(row["broad_spectrum_used"]))
sim_reserved = c2.checkbox("Use Reserved ABX", value=bool(row["reserved_abx_used"]))
sim_icu = c3.checkbox("ICU Admission", value=bool(row["icu_admission"]))

sim_input = row[FEATURES].copy()
sim_input["broad_spectrum_used"] = int(sim_broad)
sim_input["reserved_abx_used"] = int(sim_reserved)
sim_input["icu_admission"] = int(sim_icu)

sim_X = pd.DataFrame([sim_input])
sim_pred = model.predict_proba(sim_X).max()

st.metric("Simulated Risk Probability", round(sim_pred, 2))

# ===============================
# 3️⃣ INFECTION SPREAD RISK
# ===============================
st.markdown("### 🏥 Infection Spread Risk")

dept = row["department"]
dept_patients = df[df["department"] == dept]
high_cases = (dept_patients["amr_risk_level"] == "High").sum()

spread_risk = high_cases / len(dept_patients)

st.metric(
    "Cross-Transmission Risk",
    f"{round(spread_risk*100,1)}%"
)

# ===============================
# 4️⃣ CONFIDENCE SCORE
# ===============================
st.markdown("### 📊 Model Confidence")

probs = model.predict_proba(X)[0]
confidence = (probs.max() - probs.min()) * 100

st.metric("Decision Confidence", f"{round(confidence,1)}%")

if confidence < 60:
    st.warning("Low confidence — additional diagnostics recommended")

# ===============================
# 5️⃣ ACTION READINESS PANEL
# ===============================
st.markdown("### 🧾 Action Readiness Panel")

actions = []

if row["amr_risk_prob"] >= 0.7:
    actions = [
        "☑️ Isolate patient immediately",
        "☑️ Notify infection control team",
        "☑️ Perform environmental disinfection",
        "☐ Review antibiotic regimen",
        "☐ Obtain culture sample"
    ]
elif row["amr_risk_prob"] >= 0.4:
    actions = [
        "☑️ Close monitoring",
        "☐ Review antibiotics",
        "☐ Repeat labs in 24h"
    ]
else:
    actions = [
        "☑️ Standard precautions",
        "☑️ Continue current treatment"
    ]

for a in actions:
    st.write(a)


# ===============================
# MODEL PREDICTION
# ===============================
FEATURES = [
    "broad_spectrum_used",
    "reserved_abx_used",
    "antibiotic_switches",
    "icu_admission",
    "fever",
    "wbc_high",
    "amr_risk_prob"
]

X = pd.DataFrame([row[FEATURES]])
prediction = model.predict(X)[0]   # ✅ هذا هو السطر الصحيح

p1, p2 = st.columns(2)
p1.metric("Predicted AMR Risk", prediction)
p2.metric("Risk Probability", row["amr_risk_prob"])

st.subheader("Doctor Notes")
st.write(row["doctor_note"])

# ===============================
# ASSIGN DEPARTMENT
# ===============================
def assign_department(r):
    if r["icu_admission"] == 1:
        return "ICU"
    elif r["amr_risk_prob"] >= 0.7:
        return "Infectious Diseases"
    elif r["amr_risk_prob"] >= 0.4:
        return "General Ward"
    else:
        return "Outpatient"

df["department"] = df.apply(assign_department, axis=1)

# ===============================
# HOSPITAL MAP
# ===============================
HOSPITAL_MAP = {
    "ER": ["ER-1", "ER-2", "ER-Corridor"],
    "ICU": ["ICU-1", "ICU-2", "ICU-Corridor"],
    "Infectious Diseases": ["ID-1", "ID-2", "ID-Corridor"],
    "General Ward": ["GW-1", "GW-2", "GW-Corridor"],
    "Outpatient": ["OP-1", "OP-2", "OP-Corridor"]
}

def assign_room(r):
    dept = r["department"]
    rooms = HOSPITAL_MAP.get(dept, ["ER-1"])
    return rooms[hash(r["patient_uid"]) % len(rooms)]

df["room"] = df.apply(assign_room, axis=1)

# ===============================
# SUMMARY
# ===============================
dept_summary = (
    df.groupby("department")
    .agg(
        avg_risk=("amr_risk_prob", "mean"),
        high_cases=("amr_risk_level", lambda x: (x == "High").sum()),
        patients=("patient_uid", "count")
    )
    .reset_index()
)

# ===============================
# TABS
# ===============================
tab1, tab2, tab3, tab4 = st.tabs([
    "🗺️ Hospital Map",
    "📊 Dashboard",
    "🧠 Explainable AI",
    "🧾 Instructions"
])

# ===============================
# TAB 1
# ===============================
with tab1:
    fig1 = px.treemap(
        df,
        path=["department", "room"],
        values="amr_risk_prob",
        color="amr_risk_prob",
        color_continuous_scale=["green", "yellow", "red"]
    )
    st.plotly_chart(fig1, use_container_width=True)

# ===============================
# TAB 2
# ===============================
with tab2:
    fig2 = px.bar(
        dept_summary,
        x="avg_risk",
        y="department",
        orientation="h",
        color="avg_risk",
        color_continuous_scale="Reds"
    )
    st.plotly_chart(fig2, use_container_width=True)

# ===============================
# TAB 3 (Explainable AI)
# ===============================
with tab3:
    importance = pd.DataFrame({
        "Feature": FEATURES,
        "Importance": model.feature_importances_
    }).sort_values(by="Importance", ascending=False)

    fig3 = px.bar(
        importance,
        x="Importance",
        y="Feature",
        orientation="h"
    )
    st.plotly_chart(fig3, use_container_width=True)

# ===============================
# TAB 4 (Instructions)
# ===============================
with tab4:
    if row["amr_risk_prob"] >= 0.7:
        st.error("Immediate isolation & infection control required")
    elif row["amr_risk_prob"] >= 0.4:
        st.warning("Close monitoring & review antibiotics")
    else:
        st.success("Standard precautions")

st.caption("© 2026 ResistAI Sentinel")


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import plotly.express as px
import numpy as np
import uuid
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas

# ===============================
# PAGE CONFIG
# ===============================
st.set_page_config(page_title="ResistAI Sentinel", layout="wide")

# ===============================
# LOAD DATA & MODEL
# ===============================
df = pd.read_csv("data/amr_dataset.csv")
model = joblib.load("ml/amr_model.pkl")

df["patient_uid"] = df["patient_id"].apply(
    lambda _: f"MRN-{uuid.uuid4().hex[:8].upper()}"
)

# ===============================
# HEADER
# ===============================
st.title("🛡️ ResistAI Sentinel")
st.caption("Explainable AMR Clinical Decision Support System")

# ===============================
# EXECUTIVE DASHBOARD
# ===============================
c1, c2, c3, c4 = st.columns(4)
c1.metric("Total Patients", len(df))
c2.metric("High Risk Patients", (df["amr_risk_level"] == "High").sum())
c3.metric("ICU Admissions", df["icu_admission"].sum())
c4.metric("Avg Risk", round(df["amr_risk_prob"].mean(), 2))

# ===============================
# ASSIGN DEPARTMENT
# ===============================
def assign_department(r):
    if r["icu_admission"]:
        return "ICU"
    elif r["amr_risk_prob"] >= 0.7:
        return "Infectious Diseases"
    elif r["amr_risk_prob"] >= 0.4:
        return "General Ward"
    else:
        return "Outpatient"

df["department"] = df.apply(assign_department, axis=1)

# ===============================
# PATIENT SELECTION
# ===============================
st.markdown("## 🧑‍⚕️ Patient Analysis")
pid = st.selectbox("Select Patient (Medical ID)", df["patient_uid"])
row = df[df["patient_uid"] == pid].iloc[0]

FEATURES = [
    "broad_spectrum_used",
    "reserved_abx_used",
    "antibiotic_switches",
    "icu_admission",
    "fever",
    "wbc_high",
    "amr_risk_prob"
]

X = pd.DataFrame([row[FEATURES]])
prediction = model.predict(X)[0]

# ===============================
# TABS
# ===============================
tab1, tab2, tab3, tab4, tab5, tab6 = st.tabs([
    "🗺️ Hospital Heatmap",
    "🧠 Risk Trend",
    "🧭 What-If Simulator",
    "🏥 Spread Risk",
    "📊 Explainable AI",
    "🧾 Actions & Reports"
])

# =====================================================
# TAB 1 — HOSPITAL HEATMAP
# =====================================================
with tab1:
    fig = px.treemap(
        df,
        path=["department"],
        values="amr_risk_prob",
        color="amr_risk_prob",
        color_continuous_scale=["green", "yellow", "red"]
    )
    st.plotly_chart(fig, use_container_width=True)

# =====================================================
# TAB 2 — RISK TREND + ALERT
# =====================================================
with tab2:
    np.random.seed(0)
    history = np.clip(
        row["amr_risk_prob"] + np.random.normal(0, 0.05, 5), 0, 1
    )

    trend_score = history[-3:].mean() - history[:3].mean()
    if trend_score > 0.05:
        st.error("🔴 Rising Risk – Early Warning")
        trend_label = "Rising"
    elif trend_score < -0.05:
        st.success("🟢 Decreasing Risk")
        trend_label = "Decreasing"
    else:
        st.warning("🟡 Stable Risk")
        trend_label = "Stable"

    st.line_chart(history)

# =====================================================
# TAB 3 — WHAT-IF SIMULATOR
# =====================================================
with tab3:
    b = st.checkbox("Broad-Spectrum Antibiotic", bool(row["broad_spectrum_used"]))
    r = st.checkbox("Reserved Antibiotic", bool(row["reserved_abx_used"]))
    i = st.checkbox("ICU Admission", bool(row["icu_admission"]))

    sim = row[FEATURES].copy()
    sim["broad_spectrum_used"] = int(b)
    sim["reserved_abx_used"] = int(r)
    sim["icu_admission"] = int(i)

    sim_pred = model.predict_proba(pd.DataFrame([sim])).max()
    st.metric("Simulated Risk Probability", round(sim_pred, 2))

# =====================================================
# TAB 4 — INFECTION SPREAD RISK
# =====================================================
with tab4:
    dept = row["department"]
    dept_df = df[df["department"] == dept]
    spread = (dept_df["amr_risk_level"] == "High").sum() / len(dept_df)
    st.metric("Cross-Transmission Risk", f"{round(spread*100,1)}%")

# =====================================================
# TAB 5 — EXPLAINABLE AI
# =====================================================
with tab5:
    importance = pd.DataFrame({
        "Feature": FEATURES,
        "Importance": model.feature_importances_
    }).sort_values(by="Importance")

    fig = px.bar(
        importance,
        x="Importance",
        y="Feature",
        orientation="h"
    )
    st.plotly_chart(fig, use_container_width=True)

# =====================================================
# TAB 6 — ACTIONS, TIMELINE & PDF
# =====================================================
with tab6:
    # Timeline
    st.subheader("⏱️ Patient Movement Timeline")
    if row["amr_risk_prob"] < 0.4:
        timeline = ["Outpatient", "General Ward"]
    elif row["amr_risk_prob"] < 0.7:
        timeline = ["ER", "General Ward", "Infectious Diseases"]
    else:
        timeline = ["ER", "General Ward", "ICU"]

    st.table(pd.DataFrame({"Day": range(1, len(timeline)+1), "Department": timeline}))

    # Clinical Recommendation
    st.subheader("🧠 Clinical Recommendation")
    if row["amr_risk_prob"] >= 0.7:
        recommendation = (
            "High and increasing likelihood of antimicrobial resistance. "
            "Immediate isolation, antibiotic review, and infection control intervention are required."
        )
        st.error(recommendation)
    elif row["amr_risk_prob"] >= 0.4:
        recommendation = (
            "Moderate AMR risk detected. Close monitoring and review of antibiotic therapy are advised."
        )
        st.warning(recommendation)
    else:
        recommendation = (
            "Low AMR risk. Standard precautions and continued observation are appropriate."
        )
        st.success(recommendation)

    # PDF Report
    if st.button("📄 Generate Executive PDF"):
        pdf_path = f"/tmp/{row['patient_uid']}_AMR_Report.pdf"
        c = canvas.Canvas(pdf_path, pagesize=A4)

        c.setFont("Helvetica-Bold", 14)
        c.drawString(50, 800, "AMR Executive Clinical Report")

        c.setFont("Helvetica", 11)
        c.drawString(50, 770, f"Patient ID: {row['patient_uid']}")
        c.drawString(50, 750, f"Department: {row['department']}")
        c.drawString(50, 730, f"Risk Level: {row['amr_risk_level']}")
        c.drawString(50, 710, f"Risk Probability: {row['amr_risk_prob']}")

        text = c.beginText(50, 680)
        for line in recommendation.split(". "):
            text.textLine(line)
        c.drawText(text)
        c.save()

        with open(pdf_path, "rb") as f:
            st.download_button(
                "⬇️ Download PDF",
                f,
                file_name=f"{row['patient_uid']}_AMR_Report.pdf",
                mime="application/pdf"
            )

st.caption("© 2026 ResistAI Sentinel")


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import plotly.express as px
import numpy as np
import uuid, time
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas

st.set_page_config(layout="wide", page_title="ResistAI Sentinel")

# ===============================
# LOAD DATA
# ===============================
df = pd.read_csv("data/amr_dataset.csv")
model = joblib.load("ml/amr_model.pkl")

df["patient_uid"] = df["patient_id"].apply(
    lambda _: f"MRN-{uuid.uuid4().hex[:8].upper()}"
)

# ===============================
# HEADER
# ===============================
st.title("🛡️ ResistAI Sentinel")
st.caption("Dynamic AMR Clinical Decision Support System")

# ===============================
# EXECUTIVE METRICS
# ===============================
c1, c2, c3, c4 = st.columns(4)
c1.metric("Patients", len(df))
c2.metric("High Risk", (df["amr_risk_level"]=="High").sum())
c3.metric("ICU", df["icu_admission"].sum())
c4.metric("Avg Risk", round(df["amr_risk_prob"].mean(),2))

# ===============================
# ASSIGN DEPARTMENT
# ===============================
def assign_department(r):
    if r["icu_admission"]: return "ICU"
    if r["amr_risk_prob"]>=0.7: return "Infectious Diseases"
    if r["amr_risk_prob"]>=0.4: return "General Ward"
    return "Outpatient"

df["department"] = df.apply(assign_department, axis=1)

# ===============================
# SELECT PATIENT
# ===============================
pid = st.selectbox("Select Patient (Medical ID)", df["patient_uid"])
row = df[df["patient_uid"]==pid].iloc[0]

FEATURES = [
    "broad_spectrum_used","reserved_abx_used","antibiotic_switches",
    "icu_admission","fever","wbc_high","amr_risk_prob"
]

# ===============================
# SESSION STATE (FOR CHANGE)
# ===============================
if "recalc" not in st.session_state:
    st.session_state.recalc = 0

if st.button("🔄 Recalculate Analysis"):
    st.session_state.recalc += 1

# ===============================
# TABS
# ===============================
tab1,tab2,tab3,tab4,tab5,tab6 = st.tabs([
    "🗺️ Heatmap","🧠 Risk Trend","🧭 What-If",
    "🏥 Spread Risk","📊 Explainable AI","🧾 Actions & PDF"
])

# ===============================
# TAB 1 — HEATMAP
# ===============================
with tab1:
    fig = px.treemap(
        df, path=["department"],
        values="amr_risk_prob",
        color="amr_risk_prob",
        color_continuous_scale=["green","yellow","red"]
    )
    st.plotly_chart(fig, use_container_width=True)

# ===============================
# TAB 2 — RISK TREND (DYNAMIC)
# ===============================
with tab2:
    history = np.clip(
        row["amr_risk_prob"]
        + np.random.normal(0,0.05+0.01*st.session_state.recalc,5)
        + (time.time()%5)*0.01,
        0,1
    )
    trend = history[-3:].mean() - history[:3].mean()

    if trend>0.05:
        st.error("🔴 Rising Risk (Early Warning)")
        trend_label="Rising"
    elif trend<-0.05:
        st.success("🟢 Decreasing Risk")
        trend_label="Decreasing"
    else:
        st.warning("🟡 Stable Risk")
        trend_label="Stable"

    st.line_chart(history)

# ===============================
# TAB 3 — WHAT IF (REAL CHANGE)
# ===============================
with tab3:
    b = st.checkbox("Broad-Spectrum ABX", bool(row["broad_spectrum_used"]))
    r = st.checkbox("Reserved ABX", bool(row["reserved_abx_used"]))
    i = st.checkbox("ICU Admission", bool(row["icu_admission"]))

    sim = pd.DataFrame([{
        "broad_spectrum_used": int(b),
        "reserved_abx_used": int(r),
        "antibiotic_switches": row["antibiotic_switches"] + (1 if r else 0),
        "icu_admission": int(i),
        "fever": row["fever"],
        "wbc_high": row["wbc_high"],
        "amr_risk_prob": row["amr_risk_prob"]
    }])

    sim_pred = model.predict_proba(sim).max()
    st.metric("Simulated Risk", round(sim_pred,2))

# ===============================
# TAB 4 — SPREAD RISK
# ===============================
with tab4:
    dept_df = df[df["department"]==row["department"]]
    spread = (dept_df["amr_risk_level"]=="High").sum()/len(dept_df)
    st.metric("Cross-Transmission Risk", f"{round(spread*100,1)}%")

# ===============================
# TAB 5 — EXPLAINABLE AI
# ===============================
with tab5:
    imp = pd.DataFrame({
        "Feature": FEATURES,
        "Importance": model.feature_importances_
    }).sort_values(by="Importance")
    fig = px.bar(imp, x="Importance", y="Feature", orientation="h")
    st.plotly_chart(fig, use_container_width=True)

# ===============================
# TAB 6 — ACTIONS, TIMELINE, PDF
# ===============================
with tab6:
    # Timeline
    if row["amr_risk_prob"]<0.4:
        timeline=["Outpatient","General Ward"]
    elif row["amr_risk_prob"]<0.7:
        timeline=["ER","General Ward","Infectious Diseases"]
    else:
        timeline=["ER","General Ward","ICU"]

    st.subheader("⏱️ Patient Movement Timeline")
    st.table(pd.DataFrame({"Day":range(1,len(timeline)+1),"Department":timeline}))

    # Recommendation
    st.subheader("🧠 Clinical Recommendation")
    if row["amr_risk_prob"]>=0.7:
        rec=("High and increasing likelihood of antimicrobial resistance. "
             "Immediate isolation, antibiotic review, and infection control required.")
        st.error(rec)
    elif row["amr_risk_prob"]>=0.4:
        rec=("Moderate AMR risk detected. Close monitoring and review of antibiotics advised.")
        st.warning(rec)
    else:
        rec=("Low AMR risk. Standard precautions and continued observation appropriate.")
        st.success(rec)

    # Alert
    if trend_label=="Rising" and row["amr_risk_prob"]>=0.6:
        st.error("🚨 AUTOMATIC ALERT: Rising AMR Risk")

    # PDF
    if st.button("📄 Generate PDF Report"):
        path=f"/tmp/{row['patient_uid']}_AMR_Report.pdf"
        c=canvas.Canvas(path,pagesize=A4)
        c.setFont("Helvetica-Bold",14)
        c.drawString(50,800,"AMR Executive Report")
        c.setFont("Helvetica",11)
        c.drawString(50,770,f"Patient ID: {row['patient_uid']}")
        c.drawString(50,750,f"Department: {row['department']}")
        c.drawString(50,730,f"Risk Probability: {row['amr_risk_prob']}")
        text=c.beginText(50,700)
        for l in rec.split(". "): text.textLine(l)
        c.drawText(text); c.save()

        with open(path,"rb") as f:
            st.download_button("⬇️ Download PDF",f,
                file_name=f"{row['patient_uid']}_AMR_Report.pdf",
                mime="application/pdf")

st.caption("© 2026 ResistAI Sentinel")


In [ ]:
mkdir assets

In [ ]:
mkdir -p assets

In [ ]:
import os

os.makedirs("assets", exist_ok=True)

svg_content = """
<svg width="900" height="400" xmlns="http://www.w3.org/2000/svg">

  <rect id="ER_1" x="20" y="50" width="250" height="120" fill="#cccccc" stroke="#000"/>
  <text x="140" y="120" font-size="20" text-anchor="middle" fill="#000">ER</text>

  <rect id="ICU_1" x="320" y="50" width="250" height="120" fill="#cccccc" stroke="#000"/>
  <text x="445" y="120" font-size="20" text-anchor="middle" fill="#000">ICU</text>

  <rect id="Ward_1" x="620" y="50" width="250" height="120" fill="#cccccc" stroke="#000"/>
  <text x="745" y="120" font-size="20" text-anchor="middle" fill="#000">Ward</text>

  <rect x="20" y="220" width="850" height="40" fill="#e0e0e0"/>
  <text x="445" y="245" font-size="14" text-anchor="middle">Main Corridor</text>

</svg>
"""

with open("assets/hospital_floor.svg", "w") as f:
    f.write(svg_content)

print("✅ hospital_floor.svg created successfully")


In [ ]:
# =========================================================
# ResistAI Sentinel – Unified Live AMR Decision System
# =========================================================

import streamlit as st
import pandas as pd
import numpy as np
import joblib
from datetime import datetime

# -------------------------------
# Page Config
# -------------------------------
st.set_page_config(page_title="ResistAI Sentinel", layout="wide")

st.title("🧠 ResistAI Sentinel")
st.caption("Unified AMR Decision Support System")

# -------------------------------
# Load Data & Model
# -------------------------------
@st.cache_data
def load_data():
    return pd.read_csv("data/amr_dataset.csv")

@st.cache_resource
def load_model():
    return joblib.load("ml/amr_model.pkl")

df = load_data()
model = load_model()

# -------------------------------
# Normalization
# -------------------------------
df["patient_uid"] = df.get("patient_uid", df["patient_id"])

df["dept_norm"] = df["department"].replace({
    "ER": "ER",
    "Emergency": "ER",
    "ICU": "ICU",
    "Ward A": "Ward",
    "Ward B": "Ward",
    "General Ward": "Ward",
    "Outpatient": "Outpatient"
})

FEATURES = [
    "broad_spectrum_used",
    "reserved_abx_used",
    "antibiotic_switches",
    "icu_admission",
    "fever",
    "wbc_high"
]

# -------------------------------
# Sidebar – Patient
# -------------------------------
st.sidebar.header("👤 Patient")

selected_patient = st.sidebar.selectbox(
    "Select Patient",
    df["patient_uid"].unique()
)

base_df = df.copy()
row_idx = base_df[base_df["patient_uid"] == selected_patient].index[0]

filtered_df = base_df.copy()

# -------------------------------
# What-If Simulator
# -------------------------------
st.sidebar.header("🧪 What-If Simulator")

if st.sidebar.checkbox("Change Antibiotic"):
    filtered_df.loc[row_idx, ["broad_spectrum_used","reserved_abx_used"]] = 0

if st.sidebar.checkbox("Transfer to ICU"):
    filtered_df.loc[row_idx, "icu_admission"] = 1
    filtered_df.loc[row_idx, "dept_norm"] = "ICU"

if st.sidebar.checkbox("Clinical Improvement"):
    filtered_df.loc[row_idx, ["fever","wbc_high"]] = 0

# -------------------------------
# Live Risk Prediction
# -------------------------------
def predict_risk(row):
    X = pd.DataFrame([row[FEATURES]])
    X["amr_risk_prob"] = 0.0
    X = X[model.feature_names_in_]
    proba = model.predict_proba(X)[0]
    return proba.max(), model.classes_[proba.argmax()]

risk_score, risk_class = predict_risk(filtered_df.loc[row_idx])

# -------------------------------
# Metrics
# -------------------------------
c1, c2, c3 = st.columns(3)
c1.metric("AMR Risk Score", f"{risk_score:.2f}")
c2.metric("Risk Level", risk_class)
c3.metric("Department", filtered_df.loc[row_idx,"dept_norm"])

# -------------------------------
# Explainable AI
# -------------------------------
st.subheader("🧠 Why this decision?")

reasons = []
r = filtered_df.loc[row_idx]

if r["reserved_abx_used"]: reasons.append("Reserved antibiotic exposure")
if r["broad_spectrum_used"]: reasons.append("Broad-spectrum antibiotic")
if r["icu_admission"]: reasons.append("ICU admission")
if r["antibiotic_switches"] >= 1: reasons.append("Multiple antibiotic switches")
if r["fever"] and r["wbc_high"]: reasons.append("Active infection indicators")

for rs in reasons:
    st.warning(rs)

if not reasons:
    st.success("No major AMR drivers detected")

# -------------------------------
# Live Risk for All Patients
# -------------------------------
def compute_live_risk(df_):
    df_ = df_.copy()
    df_["live_risk"] = df_.apply(
        lambda r: predict_risk(r)[0],
        axis=1
    )
    return df_

live_df = compute_live_risk(filtered_df)

# -------------------------------
# Spread Risk (Department)
# -------------------------------
st.subheader("🏥 Infection Spread Risk")

spread_df = (
    live_df.groupby("dept_norm")["live_risk"]
    .agg(["mean","count"])
    .reset_index()
)

spread_df["spread_risk"] = spread_df["mean"] * np.log1p(spread_df["count"])

st.dataframe(spread_df)

# -------------------------------
# Timeline (Simulated)
# -------------------------------
st.subheader("⏱️ Patient Movement Timeline")

timeline = [
    ("ER", "08:30"),
    ("ICU", "12:00"),
    ("Ward", "18:00")
]

timeline_df = pd.DataFrame(timeline, columns=["Department","Time"])
st.table(timeline_df)

# -------------------------------
# Hospital Floor Plan (SVG)
# -------------------------------
def risk_color(r):
    if r >= 0.7: return "#d73027"
    if r >= 0.4: return "#fc8d59"
    return "#1a9850"

dept_risk = live_df.groupby("dept_norm")["live_risk"].mean().to_dict()

st.subheader("🗺️ Hospital Floor Plan")

svg = f"""
<svg width="900" height="320" xmlns="http://www.w3.org/2000/svg">
<rect x="30" y="50" width="240" height="120" fill="{risk_color(dept_risk.get('ER',0))}"/>
<text x="150" y="120" text-anchor="middle" fill="#fff">ER</text>

<rect x="330" y="50" width="240" height="120" fill="{risk_color(dept_risk.get('ICU',0))}"/>
<text x="450" y="120" text-anchor="middle" fill="#fff">ICU</text>

<rect x="630" y="50" width="240" height="120" fill="{risk_color(dept_risk.get('Ward',0))}"/>
<text x="750" y="120" text-anchor="middle" fill="#fff">Ward</text>
</svg>
"""

st.markdown(svg, unsafe_allow_html=True)

# -------------------------------
# Executive Report (Text PDF-like)
# -------------------------------
st.subheader("🧾 Executive Report")

report_text = f"""
Patient: {selected_patient}
Risk Score: {risk_score:.2f}
Risk Level: {risk_class}
Department: {filtered_df.loc[row_idx,'dept_norm']}
Generated: {datetime.now()}
"""

st.text(report_text)

st.download_button(
    "⬇️ Download Report (TXT)",
    report_text,
    file_name="AMR_Report.txt"
)

# -------------------------------
# END
# =========================================================


In [ ]:
!pkill -9 -f streamlit
!pkill -9 -f ngrok
!rm -f streamlit.log

In [ ]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0


In [ ]:
!nohup streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > streamlit.log 2>&1 &

In [ ]:
from pyngrok import ngrok
ngrok.kill()
url = ngrok.connect(8501, "http")
print("OPEN THIS URL:")
print(url)
